# Lab 01-01 · Functions, Models and Units

Companion to chapter [01-01](../../chapters/01-01-functions-models-and-units.md). Handout: [README.md](README.md).

Run the cells in order. Cells marked **🔮 Predict** ask you to write a prediction *before* running.

In [ ]:
import math, sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
sys.path.insert(0, str(Path.cwd().parents[3] / "tools"))
try:
    from plotstyle import use_style; use_style()
except ImportError:
    pass
k_B, q = 1.380649e-23, 1.602176634e-19   # exact SI values

## Part 1 — Function explorer

The same four functions on linear, semilog and log–log axes. Change the parameters (`k`, `n`, `tau`) and re-run.
Notice which family becomes a straight line on which axes.

In [ ]:
k, n, tau = 2.0, 1.5, 1.0
x = np.linspace(0.05, 5, 400)
fams = {"linear  y = kx": k * x,
        "power   y = k x^n": k * x**n,
        "exp     y = k e^(-x/tau)": k * np.exp(-x / tau),
        "log     y = ln x + 2": np.log(x) + 2}
fig, axes = plt.subplots(1, 3, figsize=(12, 3.6))
for name, y in fams.items():
    axes[0].plot(x, y, label=name)
    m = y > 0
    axes[1].semilogy(x[m], y[m], label=name)
    axes[2].loglog(x[m], y[m], label=name)
for ax, t in zip(axes, ["linear axes", "semilog (log y)", "log-log"]):
    ax.set_title(t); ax.set_xlabel("x")
axes[0].legend(fontsize=7)
plt.tight_layout(); plt.show()
# a power law's slope on log-log axes is its exponent:
slope = np.polyfit(np.log(x), np.log(k * x**n), 1)[0]
print(f"log-log slope of k x^n = {slope:.4f} (n = {n})")
assert abs(slope - n) < 1e-9

## Part 2 — Decibel calculator

Helper functions for the conversions of chapter §3.6. The example chain: −40 dBm input, then +20 dB, −3 dB, +15 dB and a −6 dB attenuator.

**🔮 Predict** the output power in dBm and in µW before running.

In [ ]:
def db_power(ratio):   return 10 * math.log10(ratio)
def db_amp(ratio):     return 20 * math.log10(ratio)
def dbm(p_watts):      return 10 * math.log10(p_watts / 1e-3)
def watts(p_dbm):      return 1e-3 * 10 ** (p_dbm / 10)

p_in = -40.0                       # dBm
chain = [20, -3, 15, -6]           # dB
p_out = p_in + sum(chain)
print(f"total gain     = {sum(chain)} dB = x{10**(sum(chain)/10):.1f} in power")
print(f"output power   = {p_out:.1f} dBm = {watts(p_out)*1e6:.2f} uW")
assert p_out == -14 and abs(watts(p_out) * 1e6 - 39.81) < 0.005
# landmarks
for r in [2, 4, 10, 100, 0.5]:
    print(f"power ratio {r:>6}: {db_power(r):6.2f} dB   amplitude ratio {r:>6}: {db_amp(r):6.2f} dB")

## Part 3 — The diode on semilog axes

$I = I_s(e^{V/V_T} - 1)$ with $V_T = kT/q$. On semilog axes the forward curve is a straight line, with slope one decade per $V_T\ln 10$.

**🔮 Predict:** at 300 K the slope is 59.5 mV/decade. What is it at 360 K? (We hold $I_s$ fixed here; in a real diode $I_s$ rises steeply with temperature, see [09-06](../../../09-semiconductor-devices/chapters/09-06-pn-junction-iv-and-capacitance.md).)

In [ ]:
Is = 1e-14
V = np.linspace(0.3, 0.8, 300)
fig, ax = plt.subplots(figsize=(6, 3.8))
for T in [300, 360]:
    VT = k_B * T / q
    I = Is * (np.exp(V / VT) - 1)
    ax.semilogy(V, I, label=f"T = {T} K")
    print(f"T = {T} K: V_T = {VT*1e3:.2f} mV, slope = {VT*math.log(10)*1e3:.2f} mV/decade")
ax.set_xlabel("V (V)"); ax.set_ylabel("I (A)"); ax.legend(); ax.set_title("Ideal diode, fixed I_s")
plt.show()
VT360 = k_B * 360 / q
assert abs(VT360 * 1e3 - 31.02) < 0.005 and abs(VT360 * math.log(10) * 1e3 - 71.43) < 0.005

## Part 4 — Fitting models to data

First the clean synthetic data of Example 01-01.5 ($P = 0.6V^3$). Then the same law with 5% random multiplicative noise, refitted many times, to show how much the fitted exponent scatters.

**🔮 Predict:** with 5% noise on five points, will the fitted exponent usually land within ±0.1 of 3, or within ±0.5?

In [ ]:
Vd = np.array([0.6, 0.7, 0.8, 0.9, 1.0])
Pd = np.array([0.130, 0.206, 0.307, 0.437, 0.600])
n_fit, lnk = np.polyfit(np.log(Vd), np.log(Pd), 1)
print(f"clean data: n = {n_fit:.3f}, k = {math.exp(lnk):.3f}")
assert abs(n_fit - 2.99) < 0.005

rng = np.random.default_rng(1)
fits = []
for _ in range(2000):
    noisy = 0.6 * Vd**3 * (1 + 0.05 * rng.standard_normal(Vd.size))
    fits.append(np.polyfit(np.log(Vd), np.log(noisy), 1)[0])
fits = np.array(fits)
print(f"noisy fits: mean n = {fits.mean():.3f}, std = {fits.std():.3f}")
print(f"fraction within ±0.1 of 3: {np.mean(abs(fits - 3) < 0.1):.0%}; within ±0.5: {np.mean(abs(fits - 3) < 0.5):.0%}")
plt.figure(figsize=(6, 3.5)); plt.hist(fits, bins=50)
plt.xlabel("fitted exponent n"); plt.ylabel("count"); plt.title("5 points, 5% noise"); plt.show()

**Exponential decay.** Fit $v = v_0 e^{-t/\tau}$ to the data of Problem 01-01.13 with a semilog line fit.

In [ ]:
t = np.arange(5.0)
v = np.array([4.000, 2.054, 1.054, 0.541, 0.278])
s, c = np.polyfit(t, np.log(v), 1)
print(f"tau = {-1/s:.3f} ms, v0 = {math.exp(c):.3f} V")
plt.figure(figsize=(5, 3.5)); plt.semilogy(t, v, "o", label="data")
tt = np.linspace(0, 6, 100); plt.semilogy(tt, math.exp(c) * np.exp(s * tt), label="fit")
plt.xlabel("t (ms)"); plt.ylabel("v (V)"); plt.legend(); plt.show()

## Your turn

1. In Part 2, add a 3 dB splitter and a 30 dB amplifier to the chain. Predict the output first.
2. In Part 4, change the noise to 20%. How does the scatter of the exponent change? How many points would you need to get back to the 5% level of scatter?